In [40]:
%load_ext autoreload
%autoreload 2
import os

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

print(os.getcwd()) # should end in /medjudge-audit

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/Users/berniceyan/medjudge-audit


In [41]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf

a = pd.read_json("results/grades_track_a.jsonl", lines=True)
v1 = a[(a.variant == "v1_official") & a.grade.notna()].copy()

v1["judge_met"] = v1.grade.astype(bool)
v1["phys_met"]  = v1.physician_label.astype(bool)
v1["log_len"]   = np.log(v1.response_len)
print(f"{len(v1)} graded criteria, {v1.judge_model.nunique()} judges (v1_official)")


4784 graded criteria, 3 judges (v1_official)


## 1. The naive (symmetric) test

Regress "judge disagreed with physician" on log length. 

In [42]:
v1["disagree"] = (v1.judge_met != v1.phys_met).astype(int)
fit0 = smf.logit("disagree ~ log_len", data=v1).fit(disp=0)
print(f"disagree ~ log_len:  coef = {fit0.params['log_len']:+.4f}  p = {fit0.pvalues['log_len']:.3f}")


disagree ~ log_len:  coef = -0.0530  p = 0.212


"disagree" is symmetric and hides direction

## 2. Split the two error directions

- **over-credit**: judge says MET where physician says NOT met  → the verbosity-bias direction
- **under-credit**: judge says NOT met where physician says MET → judge too harsh

Compare short vs long responses. 
Note these raw rates are confounded by the base rate of "met" differing with length — the clean test is in section 3.


In [43]:
med = v1.log_len.median()
v1["over_credit"]  = ( v1.judge_met & ~v1.phys_met).astype(int)
v1["under_credit"] = (~v1.judge_met &  v1.phys_met).astype(int)

for name, m in [("short", v1.log_len <= med), ("long", v1.log_len > med)]:
    d = v1[m]
    print(f"{name:5s} (n={len(d)}): over-credit {d.over_credit.mean():.1%} | "
          f"under-credit {d.under_credit.mean():.1%}")


short (n=2392): over-credit 5.3% | under-credit 10.6%
long  (n=2392): over-credit 4.6% | under-credit 9.5%


## 3. The proper test — condition on the physician label

**Test A (verbosity bias):** among criteria the physician marked NOT met, does the judge say
"met" more often as length grows? A **positive, significant** `log_len` coefficient = verbosity bias.

**Test B:** among criteria the physician marked MET, does the judge deny credit more with length?
A negative coefficient means length also *reduces* false denials — same underlying tilt toward "met."

In [44]:
notmet = v1[~v1.phys_met].copy()
notmet["y"] = notmet.judge_met.astype(int)
met    = v1[v1.phys_met].copy()
met["y"]    = (~met.judge_met).astype(int)

fitA = smf.logit("y ~ log_len", data=notmet).fit(disp=0)
fitB = smf.logit("y ~ log_len", data=met).fit(disp=0)

print(f"A) physician NOT met (n={len(notmet)}): P(judge says met)   log_len {fitA.params['log_len']:+.3f}  p={fitA.pvalues['log_len']:.3f}")
print(f"B) physician MET     (n={len(met)}):    P(judge says not-met) log_len {fitB.params['log_len']:+.3f}  p={fitB.pvalues['log_len']:.3f}")

odds = np.exp(fitA.params['log_len'])
print(f"\n2.7x longer response (natural log) -> {odds:.2f}x the odds of a false 'met' on a failing answer")


A) physician NOT met (n=728): P(judge says met)   log_len +0.339  p=0.000
B) physician MET     (n=4056):    P(judge says not-met) log_len -0.136  p=0.010

2.7x longer response (natural log) -> 1.40x the odds of a false 'met' on a failing answer


Test A came out **+0.34, p<0.001** (positive & significant) and Test B **-0.14, p=0.01**.
Both point the same way — **length tilts the judge toward "met."** 

This is verbosity bias. It inflates false credit on failing answers and suppresses false denials on good ones; the symmetric test in section 1 was near-zero only because those two cancel.


## 4. Is it one judge or systemic?

Re-run both conditional tests per judge.

In [45]:
for judge, g in v1.groupby("judge_model"):
    nm = g[~g.phys_met].copy(); nm["y"] = nm.judge_met.astype(int)
    m  = g[g.phys_met].copy();  m["y"]  = (~m.judge_met).astype(int)
    fa = smf.logit("y ~ log_len", data=nm).fit(disp=0)
    fb = smf.logit("y ~ log_len", data=m).fit(disp=0)
    print(f"{judge:28s} not-met: {fa.params['log_len']:+.3f} (p={fa.pvalues['log_len']:.3f}) | "
          f"met: {fb.params['log_len']:+.3f} (p={fb.pvalues['log_len']:.3f})")


anthropic/claude-sonnet-4.5  not-met: +0.338 (p=0.080) | met: -0.026 (p=0.722)
google/gemini-2.5-flash      not-met: +0.314 (p=0.010) | met: -0.381 (p=0.003)
openai/gpt-4.1               not-met: +0.500 (p=0.002) | met: -0.279 (p=0.008)


All three judges show a positive not-met coefficient, so this appears to be systemic, not one bad judge.
Strongest in **gpt-4.1 (+0.50, p=0.002)**, then flash (+0.31, p=0.01); sonnet (+0.34) is the same
size but only marginal (p=0.08, smaller n). 

Notably the *best* judge by kappa (gpt-4.1) carries the *largest* length bias.

## 5. Does verbosity bias confound the Track B model gap?

If the higher-scoring model writes longer answers, part of the 0.108 gap could be length, not quality. 
Check: (a) which model is longer, (b) whether score still tracks length after controlling for the model, and (c) whether the model gap survives that control.

In [46]:
from judgeaudit.scoring import per_example_scores

b1 = pd.read_json("results/grades_track_b1.jsonl", lines=True)
ex = per_example_scores(b1)
ex = ex[(ex.judge_model=="openai/gpt-4.1") & (ex.variant=="v1_official") & (ex.run_tag=="")]

resp = pd.read_json("results/responses.jsonl", lines=True)
resp["response_len"] = resp.response.fillna("").str.len()
resp = resp.rename(columns={"model":"response_model"})[["prompt_id","response_model","response_len"]]

df = ex.merge(resp, on=["prompt_id","response_model"], how="left")
df["log_len"] = np.log(df.response_len.clip(lower=1))

print("mean length by model:\n", df.groupby("response_model").response_len.mean().round(0), "\n")
print("mean score by model:\n",  df.groupby("response_model").score.mean().round(3), "\n")

fit = smf.ols("score ~ log_len + C(response_model)", data=df).fit()
gap_col = fit.params.filter(like="C(response_model)").index[0]
print(f"score ~ log_len + C(response_model):")
print(f"  log_len coef = {fit.params['log_len']:+.4f}  p = {fit.pvalues['log_len']:.4f}")
print(f"  model gap (controlling for length) = {fit.params[gap_col]:+.4f}")


mean length by model:
 response_model
anthropic/claude-sonnet-4.5    1431.0
openai/gpt-4o-mini             1600.0
Name: response_len, dtype: float64 

mean score by model:
 response_model
anthropic/claude-sonnet-4.5    0.478
openai/gpt-4o-mini             0.370
Name: score, dtype: float64 

score ~ log_len + C(response_model):
  log_len coef = +0.0401  p = 0.0177
  model gap (controlling for length) = -0.1102


The higher-scoring model (sonnet, 0.478) is actually the *shorter* one (~1431 vs ~1600 chars), so verbosity bias is **not** inflating the gap. If anything it slightly *masks* it.

Length does independently predict score (log_len +0.04, p=0.02), confirming the bias reaches Track B, but the model gap controlling for length is **-0.110**, essentially unchanged from the raw 0.108. **The 0.108 gap is robust to length.**

### Bottom line
Verbosity bias is real and systemic across judges (strongest in gpt-4.1): longer answers get graded "met" more readily than physicians would, ~2.7x length ≈ 1.4x the odds of a false pass. It affects Track B scores but does not explain the model gap, because the winning model is the more concise one.


## 6. Self-preference (Track B)

Does a judge grade a response model more leniently than other judges do? *True* self-preference
needs a judge that also appears as a **response** model. Here the judges are gpt-4.1 and flash and
the response models are sonnet-4.5 and gpt-4o-mini — no exact overlap — so the only testable thing
is **provider-level** affinity (gpt-4.1 ↔ gpt-4o-mini, both OpenAI). The tell is the
judge × response-model interaction

In [47]:
from judgeaudit.scoring import per_example_scores

b2 = pd.read_json("results/grades_track_b2.jsonl", lines=True)
exb = per_example_scores(b2)
fit = smf.ols("score ~ C(judge_model) * C(response_model)", data=exb).fit()
inter = [i for i in fit.params.index if ":" in i]           # the interaction rows
out = pd.concat([fit.params[inter], fit.conf_int().loc[inter]], axis=1)
out.columns = ["coef", "ci_lo", "ci_hi"]
print(out.round(4))


                                                      coef   ci_lo   ci_hi
C(judge_model)[T.openai/gpt-4.1]:C(response_mod... -0.0493 -0.1294  0.0308


Interaction coef **-0.049, 95% CI [-0.129, +0.031]**. The interval **includes zero**.
With n=100 examples there is *no* evidence of provider self-preference; if anything gpt-4.1 grades the OpenAI model slightly lower, but it's indistinguishable from noise.
Takeaway: **inconclusive at this sample size and with non overlapping judges** 


## 7. Where disagreement lives (Track A)

Not all content is equally hard to grade. Stratify judge-physician disagreement by theme family,
with clustered CIs per stratum (criteria within one conversation are correlated).


In [48]:
from judgeaudit.uncertainty import cluster_bootstrap

v1["family"] = (v1.theme.str.removeprefix("cluster:")
                  .str.extract(r"^(communication|complex_responses|context_seeking|"
                               r"health_data_tasks|hedging|emergency_referrals|global_health)")[0]
                  .fillna("other"))

rows = []
for fam, g in v1.groupby("family"):
    pt, lo, hi = cluster_bootstrap(g, lambda d: d.disagree.mean(), cluster_col="prompt_id")
    rows.append({"family": fam, "disagree": pt, "lo": lo, "hi": hi, "n": len(g)})
theme_tbl = pd.DataFrame(rows).sort_values("disagree", ascending=False)
print(theme_tbl.round(3).to_string(index=False))
# fine-grained view: swap `family` for `theme` in the groupby above (34 clusters).


             family  disagree    lo    hi    n
  complex_responses     0.232 0.184 0.281  397
  health_data_tasks     0.219 0.170 0.268  471
emergency_referrals     0.188 0.148 0.232  531
    context_seeking     0.166 0.130 0.207  547
      communication     0.142 0.113 0.172  991
      global_health     0.125 0.083 0.170  400
            hedging     0.097 0.077 0.119 1447


Disagreement is highest on **complex_responses (0.23 [0.18, 0.28])** and **health_data_tasks (0.22)**, lowest on **hedging (0.10 [0.08, 0.12])**. All have non-overlapping CIs, so a real difference. 
Safety-relevant: **emergency_referrals** sits at **0.19**, meaning judges disagree with physicians on nearly 1 in 5 emergency-referral criteria.


## 8. Harm (negative-point) criteria — the safety-critical ones

Track B rows carry `points`; negative points mark harm/safety criteria (met = the model did the bad thing). We can't measure judge *accuracy* here (B rows have no physician labels), but we can measure two things: 1) how often harm criteria fire, and 2) how *reliably* judges grade them, using the B3 repeats as a consistency probe.


In [49]:
b1 = pd.read_json("results/grades_track_b1.jsonl", lines=True)
b1["harm_criterion"] = b1.points < 0
print("met-rate (fraction graded True):")
print(b1.groupby("harm_criterion").grade.mean().round(3))
print("counts:", dict(b1.groupby("harm_criterion").size()))


met-rate (fraction graded True):
harm_criterion
False    0.525
True     0.295
Name: grade, dtype: float64
counts: {False: np.int64(6822), True: np.int64(3038)}


In [50]:
# Reliability without ground truth: do harm verdicts flip more across repeats?
b3 = pd.read_json("results/grades_track_b3.jsonl", lines=True)
b3["harm_criterion"] = b3.points < 0
piv = b3.pivot_table(index=["prompt_id", "response_model", "criterion_idx", "harm_criterion"],
                     columns="run_tag", values="grade", aggfunc="first")
flip = (piv.nunique(axis=1) > 1)
print("rep flip rate (verdict changes across the 5 temp-1.0 reps):")
print(flip.groupby(level="harm_criterion").mean().round(3))


rep flip rate (verdict changes across the 5 temp-1.0 reps):
harm_criterion
False    0.122
True     0.259
dtype: float64


Harm criteria flip **25.9%** of the time across repeats vs **12.2%** for positive criteria. Judges are **~2x less consistent on exactly the safety-critical criteria**, concerning as the eval is least trustworthy where stakes are highest.

The met-rates above are descriptive only — negative-point harm criteria was met ~30% vs ~52% for positive criteria, *not* graded on accuracy since B rows have no physician labels.
